# Parte 1 — Iteración 8: min_df=1 y Vocabulario Extendido

En la iteración 7 se comprobó que `strip_accents=None` con `ngram_range=(1,7)` y `max_features=250k` (C=0.40) supera claramente la configuración anterior. El score en Kaggle fue 0.30085.

Esta iteración explora dos hipótesis adicionales:

1. **`min_df=1`**: actualmente se descartan n-grams que aparecen solo una vez. Para textos históricos con 39 clases finas, patrones ortográficos rarísimos pero específicos de una sola década podrían ser altamente discriminativos.
2. **Vocabulario más grande y rango más amplio**: aumentar `max_features` a 300k-400k y probar `ngram_range=(1,8)` para capturar secuencias más largas.

## 1. Importación de librerías

In [1]:
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

## 2. Carga de los datos

In [2]:
TRAIN_PATH      = 'train.csv'
EVAL_PATH       = 'eval.csv'
MODEL_PATH      = 'best_model_v8.joblib'
SUBMISSION_PATH = 'submission_v8.csv'

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
eval_df  = pd.read_csv(EVAL_PATH)

train_df['text'] = train_df['text'].fillna('')
eval_df['text']  = eval_df['text'].fillna('')

print('Train shape:', train_df.shape)
print('Eval shape: ', eval_df.shape)

Train shape: (31403, 2)
Eval shape:  (3490, 2)


In [4]:
train_df.head(3)

,text,decade
0,\nHonorarias ¡jubiladas. 57 \ndit.ad Pontem de...,164
1,"gone. Sus amigos , sus clientes, todo \ncuanto...",182
2,"Prefosen quemanera,e per qualesfolpechas deuan...",157


## 3. Partición de los datos

Mantenemos el mismo split de las iteraciones anteriores para que los scores sean comparables.

In [5]:
X = train_df['text']
y = train_df['decade']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

print('Train:', X_train.shape[0], '| Val:', X_val.shape[0])

Train: 25122 | Val: 6281


## 4. Diseño de experimentos

Partimos del mejor modelo de la iteración 7 y exploramos sistemáticamente el efecto de `min_df`, el tamaño del vocabulario y el rango de n-grams.

In [6]:
def build_char_model(c_value=0.40, ngram_range=(1, 7), max_features=250000,
                     min_df=2, max_df=1.0):
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents=None,
            analyzer='char',
            ngram_range=ngram_range,
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=True,
            max_features=max_features,
            dtype=np.float32,
        )),
        ('clf', LinearSVC(C=c_value)),
    ])

In [7]:
candidate_models = {
    'ref_iter7':          build_char_model(c_value=0.40, ngram_range=(1, 7), max_features=250000, min_df=2),
    'mindf1_250k':        build_char_model(c_value=0.40, ngram_range=(1, 7), max_features=250000, min_df=1),
    'mindf1_300k':        build_char_model(c_value=0.40, ngram_range=(1, 7), max_features=300000, min_df=1),
    'mindf1_400k_c030':   build_char_model(c_value=0.30, ngram_range=(1, 7), max_features=400000, min_df=1),
    'wider_1_8_mindf1':   build_char_model(c_value=0.40, ngram_range=(1, 8), max_features=350000, min_df=1),
}

## 5. Entrenamiento y comparación de candidatos

In [8]:
results = []

for name, model in candidate_models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = model.predict(X_val)
    acc   = accuracy_score(y_val, preds)
    f1    = f1_score(y_val, preds, average='macro')

    results.append({'modelo': name, 'val_accuracy': acc, 'macro_f1': f1, 'tiempo_s': round(elapsed, 1)})
    print(f'{name:<24} acc={acc:.4f}  f1={f1:.4f}  ({elapsed:.1f}s)')

MemoryError: 

In [ ]:
results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

Con el mejor rango y configuración identificados, afinamos el valor de `C`.

In [ ]:
best_name     = results_df.iloc[0]['modelo']
best_pipeline = candidate_models[best_name]
best_ngram    = best_pipeline.named_steps['tfidf'].ngram_range
best_maxf     = best_pipeline.named_steps['tfidf'].max_features
best_mindf    = best_pipeline.named_steps['tfidf'].min_df

print(f'Mejor candidato base: {best_name}')
print(f'  ngram_range={best_ngram}, max_features={best_maxf}, min_df={best_mindf}')

In [ ]:
c_candidates = {}
for c_val in [0.15, 0.20, 0.25, 0.30, 0.40, 0.50]:
    name  = f'c_{int(c_val*100):03d}'
    model = build_char_model(c_value=c_val, ngram_range=best_ngram,
                              max_features=best_maxf, min_df=best_mindf)
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    preds = model.predict(X_val)
    acc   = accuracy_score(y_val, preds)
    f1    = f1_score(y_val, preds, average='macro')

    c_candidates[name] = {'model': model, 'val_accuracy': acc, 'macro_f1': f1, 'C': c_val}
    print(f'{name}  C={c_val:.2f}  acc={acc:.4f}  f1={f1:.4f}  ({elapsed:.1f}s)')

In [ ]:
tuning_df = pd.DataFrame([
    {'C': v['C'], 'val_accuracy': v['val_accuracy'], 'macro_f1': v['macro_f1']}
    for v in c_candidates.values()
]).sort_values('val_accuracy', ascending=False)

print(tuning_df.to_string(index=False))

## 6. Evaluación del mejor modelo

In [ ]:
best_c_name  = max(c_candidates, key=lambda k: c_candidates[k]['val_accuracy'])
best_c_entry = c_candidates[best_c_name]
best_model   = best_c_entry['model']

print(f'Mejor modelo: {best_c_name}')
print(f"Val accuracy: {best_c_entry['val_accuracy']:.4f}")
print(f"Macro F1:     {best_c_entry['macro_f1']:.4f}")

In [ ]:
y_val_pred = best_model.predict(X_val)
print(classification_report(y_val, y_val_pred))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred, ax=ax, colorbar=False)
plt.title('Matriz de confusión — validación (iter8)')
plt.tight_layout()
plt.show()

## 7. Predicciones sobre eval.csv

Reentrenamos el mejor pipeline sobre el conjunto completo de entrenamiento antes de generar las predicciones finales.

In [ ]:
final_model = clone(best_model)
final_model.fit(X, y)

joblib.dump(final_model, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')

In [ ]:
eval_predictions = final_model.predict(eval_df['text']).astype(int)

submission_df = pd.DataFrame({
    'id':     eval_df['id'],
    'answer': eval_predictions,
})
submission_df.to_csv(SUBMISSION_PATH, index=False)

print(f'Submission guardado en: {SUBMISSION_PATH}')
submission_df.head()